# 🌍 Multilingual Sentiment & Emotion Analysis
**Stack:** XLM-RoBERTa · TensorFlow · HuggingFace · scikit-learn

**Languages supported:** English, Hindi, French, German, Spanish, Arabic, Chinese & 93 more

**Tasks:**
- Sentiment: Positive / Neutral / Negative
- Emotion: Joy · Anger · Fear · Sadness · Surprise · Disgust (multi-label)

---
⚡ **Before running:** Go to `Runtime → Change runtime type → T4 GPU`

## ✅ Cell 1 — Install Dependencies

In [ ]:
!pip install transformers datasets tensorflow scikit-learn seaborn matplotlib -q
print('✅ All dependencies installed!')

## ✅ Cell 2 — Imports & GPU Check

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, TFAutoModel
from datasets import load_dataset
from sklearn.metrics import (
    classification_report, f1_score,
    roc_auc_score, confusion_matrix
)
from sklearn.preprocessing import MultiLabelBinarizer
import warnings
warnings.filterwarnings('ignore')

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✅ GPU detected: {gpus[0].name}')
else:
    print('⚠️  No GPU found. Go to Runtime → Change runtime type → T4 GPU')

## ✅ Cell 3 — Load & Explore Datasets

In [ ]:
# --- Emotion dataset (GoEmotions - 28 → simplified to 6 Ekman emotions) ---
print('Loading GoEmotions dataset...')
go_emotions = load_dataset('go_emotions', 'simplified')
print(f'GoEmotions train size: {len(go_emotions["train"])}')

# --- Sentiment dataset ---
print('\nLoading TweetEval sentiment dataset...')
tweet_sent = load_dataset('tweet_eval', 'sentiment')
print(f'TweetEval train size: {len(tweet_sent["train"])}')

# Preview
print('\n--- GoEmotions sample ---')
print(go_emotions['train'][0])

print('\n--- TweetEval sample ---')
print(tweet_sent['train'][0])

## ✅ Cell 4 — Preprocess & Tokenize

In [ ]:
# Load XLM-RoBERTa tokenizer
MODEL_NAME = 'xlm-roberta-base'
MAX_LEN    = 128
BATCH_SIZE = 32

print(f'Loading tokenizer: {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('✅ Tokenizer loaded!')

# ---- Emotion preprocessing ----
# GoEmotions has 28 labels → we map to 6 Ekman emotions
EKMAN_MAP = {
    'joy':      [1, 5, 17, 19, 20, 23, 25],   # admiration,amusement,excitement,gratitude,joy,love,optimism
    'anger':    [2, 3, 16],                    # anger,annoyance,disapproval
    'fear':     [6, 14],                       # disgust,fear  (overlap handled)
    'sadness':  [9, 10, 11, 22],               # disappointment,disapproval,disgust,grief,sadness
    'surprise': [24, 26, 27],                  # surprise,realization,confusion
    'disgust':  [6],                           # disgust
}
EMOTION_LABELS = ['joy', 'anger', 'fear', 'sadness', 'surprise', 'disgust']
SENTIMENT_LABELS = ['negative', 'neutral', 'positive']

def map_to_ekman(label_ids):
    ekman = [0] * 6
    for idx, (emotion, go_ids) in enumerate(EKMAN_MAP.items()):
        if any(l in go_ids for l in label_ids):
            ekman[idx] = 1
    # Ensure at least one label
    if sum(ekman) == 0:
        ekman[0] = 1
    return ekman

def tokenize_batch(texts, max_len=MAX_LEN):
    enc = tokenizer(
        texts, truncation=True,
        padding='max_length',
        max_length=max_len,
        return_tensors='np'
    )
    return enc['input_ids'], enc['attention_mask']

# --- Build emotion dataset (subset for speed on free GPU) ---
SUBSET = 8000
print(f'\nPreparing emotion data (subset={SUBSET})...')
emo_train = go_emotions['train'].select(range(SUBSET))
emo_val   = go_emotions['validation']

emo_texts_train = emo_train['text']
emo_texts_val   = emo_val['text']
emo_labels_train = np.array([map_to_ekman(x) for x in emo_train['labels']])
emo_labels_val   = np.array([map_to_ekman(x) for x in emo_val['labels']])

emo_ids_train, emo_mask_train = tokenize_batch(emo_texts_train)
emo_ids_val,   emo_mask_val   = tokenize_batch(emo_texts_val)

# --- Build sentiment dataset ---
SENT_SUBSET = 8000
print(f'Preparing sentiment data (subset={SENT_SUBSET})...')
sent_train = tweet_sent['train'].select(range(SENT_SUBSET))
sent_val   = tweet_sent['validation']

sent_texts_train = sent_train['text']
sent_texts_val   = sent_val['text']
sent_labels_train = np.array(sent_train['label'])
sent_labels_val   = np.array(sent_val['label'])

sent_ids_train, sent_mask_train = tokenize_batch(sent_texts_train)
sent_ids_val,   sent_mask_val   = tokenize_batch(sent_texts_val)

print('✅ Data ready!')
print(f'   Emotion  train: {emo_ids_train.shape}, labels: {emo_labels_train.shape}')
print(f'   Sentiment train: {sent_ids_train.shape}, labels: {sent_labels_train.shape}')

## ✅ Cell 5 — Build the Multi-Task Model

In [ ]:
print(f'Loading XLM-RoBERTa encoder: {MODEL_NAME}...')
encoder = TFAutoModel.from_pretrained(MODEL_NAME)

# Freeze encoder weights — saves memory, trains faster on free GPU
encoder.trainable = False
print(f'Encoder frozen. Trainable params: {np.sum([np.prod(v.shape) for v in encoder.trainable_variables]):,}')

# --- Model Definition ---
input_ids      = tf.keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name='input_ids')
attention_mask = tf.keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name='attention_mask')

# Shared encoder → [CLS] token representation
outputs = encoder(input_ids, attention_mask=attention_mask)
cls_vec = outputs.last_hidden_state[:, 0, :]   # (batch, 768)
dropped = tf.keras.layers.Dropout(0.3)(cls_vec)

# Head 1 — Sentiment (3 classes, softmax)
sentiment_hidden = tf.keras.layers.Dense(256, activation='relu')(dropped)
sentiment_out    = tf.keras.layers.Dense(3, activation='softmax', name='sentiment')(sentiment_hidden)

# Head 2 — Emotion (6 classes, sigmoid — multi-label)
emotion_hidden = tf.keras.layers.Dense(256, activation='relu')(dropped)
emotion_out    = tf.keras.layers.Dense(6, activation='sigmoid', name='emotion')(emotion_hidden)

model = tf.keras.Model(
    inputs=[input_ids, attention_mask],
    outputs={'sentiment': sentiment_out, 'emotion': emotion_out}
)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=2e-4),
    loss={
        'sentiment': 'sparse_categorical_crossentropy',
        'emotion':   'binary_crossentropy'
    },
    loss_weights={'sentiment': 1.0, 'emotion': 1.0},
    metrics={
        'sentiment': ['accuracy'],
        'emotion':   [tf.keras.metrics.AUC(name='auc', multi_label=True)]
    }
)

model.summary()

## ✅ Cell 6 — Train Emotion Head

In [ ]:
print('Training emotion head...')

# Dummy sentiment labels for emotion training (model needs both outputs)
dummy_sent_train = np.zeros(len(emo_ids_train), dtype=np.int32)
dummy_sent_val   = np.zeros(len(emo_ids_val),   dtype=np.int32)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_emotion_auc', patience=2,
        restore_best_weights=True, mode='max'
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=1, verbose=1
    )
]

history_emo = model.fit(
    {'input_ids': emo_ids_train, 'attention_mask': emo_mask_train},
    {'sentiment': dummy_sent_train, 'emotion': emo_labels_train},
    validation_data=(
        {'input_ids': emo_ids_val, 'attention_mask': emo_mask_val},
        {'sentiment': dummy_sent_val, 'emotion': emo_labels_val}
    ),
    batch_size=BATCH_SIZE,
    epochs=5,
    callbacks=callbacks
)
print('✅ Emotion training complete!')

## ✅ Cell 7 — Train Sentiment Head

In [ ]:
print('Training sentiment head...')

# Dummy emotion labels for sentiment training
dummy_emo_train = np.zeros((len(sent_ids_train), 6), dtype=np.float32)
dummy_emo_val   = np.zeros((len(sent_ids_val),   6), dtype=np.float32)

callbacks_sent = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_sentiment_accuracy', patience=2,
        restore_best_weights=True, mode='max'
    )
]

history_sent = model.fit(
    {'input_ids': sent_ids_train, 'attention_mask': sent_mask_train},
    {'sentiment': sent_labels_train, 'emotion': dummy_emo_train},
    validation_data=(
        {'input_ids': sent_ids_val, 'attention_mask': sent_mask_val},
        {'sentiment': sent_labels_val, 'emotion': dummy_emo_val}
    ),
    batch_size=BATCH_SIZE,
    epochs=5,
    callbacks=callbacks_sent
)
print('✅ Sentiment training complete!')

## ✅ Cell 8 — Plot Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Emotion AUC
axes[0].plot(history_emo.history['emotion_auc'],     label='Train AUC', linewidth=2)
axes[0].plot(history_emo.history['val_emotion_auc'], label='Val AUC',   linewidth=2, linestyle='--')
axes[0].set_title('Emotion Head — AUC-ROC', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('AUC')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Sentiment Accuracy
axes[1].plot(history_sent.history['sentiment_accuracy'],     label='Train Acc', linewidth=2)
axes[1].plot(history_sent.history['val_sentiment_accuracy'], label='Val Acc',   linewidth=2, linestyle='--')
axes[1].set_title('Sentiment Head — Accuracy', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training curves saved as training_curves.png')

## ✅ Cell 9 — Evaluate on Validation Sets

In [ ]:
# --- Sentiment Evaluation ---
print('=== SENTIMENT EVALUATION ===')
sent_preds_raw = model.predict(
    {'input_ids': sent_ids_val, 'attention_mask': sent_mask_val},
    batch_size=64, verbose=0
)['sentiment']
sent_preds = sent_preds_raw.argmax(axis=1)

print(classification_report(
    sent_labels_val, sent_preds,
    target_names=SENTIMENT_LABELS
))

# Confusion matrix
cm = confusion_matrix(sent_labels_val, sent_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=SENTIMENT_LABELS, yticklabels=SENTIMENT_LABELS)
plt.title('Sentiment Confusion Matrix')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('sentiment_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Emotion Evaluation ---
print('\n=== EMOTION EVALUATION ===')
emo_preds_raw = model.predict(
    {'input_ids': emo_ids_val, 'attention_mask': emo_mask_val},
    batch_size=64, verbose=0
)['emotion']
emo_preds_bin = (emo_preds_raw > 0.4).astype(int)

f1_macro = f1_score(emo_labels_val, emo_preds_bin, average='macro', zero_division=0)
f1_micro = f1_score(emo_labels_val, emo_preds_bin, average='micro', zero_division=0)
try:
    auc = roc_auc_score(emo_labels_val, emo_preds_raw, average='macro')
except:
    auc = 0.0

print(f'F1-macro : {f1_macro:.4f}')
print(f'F1-micro : {f1_micro:.4f}')
print(f'AUC-ROC  : {auc:.4f}')
print()
print(classification_report(
    emo_labels_val, emo_preds_bin,
    target_names=EMOTION_LABELS, zero_division=0
))

## ✅ Cell 10 — Emotion Probability Heatmap

In [ ]:
# Visualize emotion probabilities for first 20 validation samples
sample_probs = emo_preds_raw[:20]
sample_texts = [t[:40] + '...' if len(t) > 40 else t for t in emo_texts_val[:20]]

plt.figure(figsize=(12, 8))
sns.heatmap(
    sample_probs,
    annot=True, fmt='.2f',
    xticklabels=EMOTION_LABELS,
    yticklabels=[f'{i+1}. {t}' for i, t in enumerate(sample_texts)],
    cmap='YlOrRd', vmin=0, vmax=1
)
plt.title('Emotion Probability Heatmap (20 validation samples)', fontsize=13)
plt.xlabel('Emotion'); plt.ylabel('Sample text')
plt.tight_layout()
plt.savefig('emotion_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Heatmap saved as emotion_heatmap.png')

## ✅ Cell 11 — Multilingual Inference Demo

In [ ]:
def predict(text: str, threshold: float = 0.4) -> dict:
    """Run sentiment + emotion inference on any language text."""
    enc = tokenizer(
        text, return_tensors='np',
        truncation=True, padding='max_length', max_length=MAX_LEN
    )
    out = model.predict(
        {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask']},
        verbose=0
    )
    sent_idx     = out['sentiment'][0].argmax()
    sent_conf    = float(out['sentiment'][0].max())
    emo_probs    = out['emotion'][0]
    detected_emo = [
        EMOTION_LABELS[i]
        for i, p in enumerate(emo_probs) if p > threshold
    ]
    return {
        'text':       text,
        'sentiment':  SENTIMENT_LABELS[sent_idx],
        'confidence': f'{sent_conf:.1%}',
        'emotions':   detected_emo if detected_emo else ['neutral'],
        'raw_probs':  {e: round(float(p), 3) for e, p in zip(EMOTION_LABELS, emo_probs)}
    }

# ---- Test across 8 languages ----
test_samples = [
    ('English',  'I absolutely love this product! It made my day so much better.'),
    ('Hindi',    'मुझे यह बिल्कुल पसंद नहीं आया, यह बहुत खराब था।'),
    ('French',   "C'est magnifique! Je suis tellement heureux aujourd'hui."),
    ('German',   'Das ist schrecklich und macht mir wirklich Angst.'),
    ('Spanish',  '¡Estoy muy emocionado con esta noticia tan sorprendente!'),
    ('Arabic',   'هذا رائع جداً وأنا سعيد للغاية بهذه النتيجة.'),
    ('Chinese',  '这个产品真的很糟糕，我非常失望和愤怒。'),
    ('Japanese', 'これは素晴らしい！とても嬉しいです。'),
]

print('='*65)
print(f'{"MULTILINGUAL INFERENCE DEMO":^65}')
print('='*65)

results = []
for lang, text in test_samples:
    result = predict(text)
    results.append({'language': lang, **result})
    print(f'\n🌐 [{lang}]')
    print(f'   Text      : {text[:55]}...' if len(text) > 55 else f'   Text      : {text}')
    print(f'   Sentiment : {result["sentiment"].upper()} ({result["confidence"]})')
    print(f'   Emotions  : {", ".join(result["emotions"])}')
    print(f'   Raw probs : {result["raw_probs"]}')

print('\n' + '='*65)
print('✅ Inference complete!')

## ✅ Cell 12 — Results Summary Table

In [ ]:
# Build a nice summary DataFrame
summary_rows = []
for r in results:
    row = {'Language': r['language'], 'Sentiment': r['sentiment'],
           'Confidence': r['confidence'], 'Emotions': ', '.join(r['emotions'])}
    row.update({f'P({e})': r['raw_probs'][e] for e in EMOTION_LABELS})
    summary_rows.append(row)

df = pd.DataFrame(summary_rows)

# Color sentiment column
def color_sent(val):
    colors = {'positive': 'background-color: #d4edda',
              'negative': 'background-color: #f8d7da',
              'neutral':  'background-color: #fff3cd'}
    return colors.get(val, '')

styled = df.style.applymap(color_sent, subset=['Sentiment'])
display(styled)

# Save to CSV
df.to_csv('multilingual_results.csv', index=False)
print('\n✅ Results saved to multilingual_results.csv')

## ✅ Cell 13 — Save the Model

In [ ]:
# Save model weights
model.save_weights('multilingual_sentiment_emotion_weights.h5')
print('✅ Model weights saved: multilingual_sentiment_emotion_weights.h5')

# Save to Google Drive (optional)
# Uncomment below to save to your Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# model.save_weights('/content/drive/MyDrive/xlm_roberta_sentiment_emotion.h5')
# print('✅ Saved to Google Drive!')

print('\n🎉 Project complete! Summary:')
print('   - Model    : XLM-RoBERTa base (multi-task, 2 heads)')
print('   - Tasks    : Sentiment (3-class) + Emotion (6-class multi-label)')
print('   - Languages: 100+ supported via XLM-RoBERTa')
print('   - Outputs  : training_curves.png, sentiment_confusion.png,')
print('                emotion_heatmap.png, multilingual_results.csv')

## ✅ Cell 14 — Try Your Own Text!

In [ ]:
# 👇 Change this text to anything you want — any language!
my_text = "I am so excited about this project, it turned out amazing!"

result = predict(my_text)
print(f'Text      : {result["text"]}')
print(f'Sentiment : {result["sentiment"].upper()} ({result["confidence"]})')
print(f'Emotions  : {", ".join(result["emotions"])}')
print()
print('Emotion breakdown:')
for emotion, prob in result['raw_probs'].items():
    bar = '█' * int(prob * 20)
    print(f'  {emotion:<10} {bar:<20} {prob:.3f}')